# 10 · Sale timing and the final cash opportunity

**Question:** Can a last-callback gate preserve profitable earlier timing while recovering otherwise unsold terminal goods?

Notebook 09 completed but lost three final coins in the first pair. Keep that negative result. This notebook tests a *new* bounded hypothesis. No resets, installs, raw-data downloads, new full seasons or automatic submissions. Earlier artifacts are read-only.

The delivered notebook is unexecuted. Reference figures describe your returned notebook-09 data; later figures require the live notebook-10 report. Neither coins nor local match scores are leaderboard ratings.

In [1]:
from pathlib import Path
import json, sys, subprocess, time, os, signal, select
import pandas as pd
import plotly.io as pio
from IPython.display import display
BASE = Path.cwd()
if not (BASE / 'run_timing.py').is_file():
    BASE = Path.home() / 'kaggriculture_sale_timing'
assert (BASE / 'run_timing.py').is_file(), 'Open the notebook in the extracted package folder.'
sys.path.insert(0, str(BASE))
from visualize_timing import diagnosis_figures, experiment_figures, style, save_dashboard
OUT = BASE / 'outputs'
OUT.mkdir(exist_ok=True)
pio.renderers.default = 'plotly_mimetype'
print('LIVE AWS WORKFLOW — previous experiments will not be rerun.')
print('Package:', BASE)

LIVE AWS WORKFLOW — previous experiments will not be rerun.
Package: /home/sagemaker-user/kaggriculture_sale_timing


## Returned result—not a current error
The retained failure files predate the successful corrected run. Only one of seven planned pairs was evaluated; the early-stop result is not a completed seven-pair trial.

In [2]:
review = json.loads((BASE/'reference/input_review.json').read_text())
old = json.loads((BASE/'reference/notebook09_report.json').read_text())
display(pd.DataFrame([{'decision': old['decision'], 'completed_pairs': old['completed_pairs'],
    'planned_pairs': old['planned_pairs'], 'coins_delta': old['primary_coin_deltas'][0]['coins_delta'],
    'worker_seconds': old['elapsed_seconds'], 'prior_tests': old['tests']['tests']}]))
print('Verified returned bundle hashes:', review['bundle_hashes_verified'])

,decision,completed_pairs,planned_pairs,coins_delta,worker_seconds,prior_tests
0,STOP_NEGATIVE_TERMINAL_EFFECT,1,7,-3.0,7.512333,86


Verified returned bundle hashes: 29


In [3]:
figures = style(diagnosis_figures(BASE/'reference'))
figures[0].show()

In [4]:
figures[1].show()
diagnosis = json.loads((BASE/'reference/diagnosis.json').read_text())
display(pd.DataFrame(diagnosis['product_deltas']))

,product,control,aligned,delta
0,EGG,172.0,172.0,0.0
1,FERTILIZER,299.0,299.0,0.0
2,TOMATO,335.0,332.0,-3.0
3,WHEAT,196.0,196.0,0.0
4,WOOL,987.0,987.0,0.0


## Mechanism and fixed intervention

The three-coin loss reconciles to four tomatoes sold one callback earlier. Unknown opponent orders are not used as policy inputs. Known demand can make waiting attractive before the endpoint, while a final deposit must be sold before the last market phase.

**Control before step 718; post-action aligned SELL quantities only at step 718.** Routing, hiring, reserve rules and farm actions stay unchanged. This threshold is determined by the game horizon, not fitted to the seed or price. The other 119 newly logged timing descriptors are diagnostic candidates, not action-changing trained features.

The run proves 22 earlier terminal-day actions per source are unchanged, recomputes the opponent at the common final state, and compares two exact terminal transitions. No earlier source simulator steps need to be regenerated. Failure of prefix equality blocks this shortcut.

In [5]:
protocol = json.loads((BASE/'PROTOCOL.json').read_text())
display(pd.DataFrame([{'worker_cap_seconds':180,'callback_cap_ms':500,'max_source_pairs':7,
    'max_research_transitions':14,'mechanics_transitions':38,'new_timing_candidates':120}]))
print('Exploratory development study. Stop at the first negative primary endpoint.')

,worker_cap_seconds,callback_cap_ms,max_source_pairs,max_research_transitions,mechanics_transitions,new_timing_candidates
0,180,500,7,14,38,120


Exploratory development study. Stop at the first negative primary endpoint.


## Run the bounded experiment
This cell calls the existing verified Python kernel as a child process. UTC heartbeats are printed. A saved completed result is reused; a failed attempt is not automatically retried.

In [6]:
cmd = [sys.executable, str(BASE/'run_timing.py'), 'run']
process = subprocess.Popen(cmd, cwd=BASE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)
started = time.monotonic()
try:
    while process.poll() is None:
        if time.monotonic() - started > 205:
            raise TimeoutError('Notebook emergency deadline; bundle diagnostics.')
        ready, _, _ = select.select([process.stdout], [], [], 1.0)
        if ready:
            line = process.stdout.readline()
            if line: print(line, end='', flush=True)
    tail = process.stdout.read()
    if tail: print(tail, end='')
    if process.returncode:
        raise RuntimeError('Experiment stopped with an error. Do not retry unchanged; run the bundle command in START_HERE.md.')
except BaseException:
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
    raise

......................................................................
----------------------------------------------------------------------
Ran 70 tests in 0.572s

OK
{"utc": "2026-09-12T00:20:08.384980+00:00", "stage": "DIAGNOSIS_RECONCILED", "cash_delta": -3.0}
{"utc": "2026-09-12T00:20:09.100662+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-12T00:20:09.138940+00:00", "stage": "MECHANICS_PASSED", "checks": 20, "interpreter_calls": 38}
{"utc": "2026-09-12T00:20:10.346020+00:00", "stage": "EPISODE_CHECKPOINT_SAVED", "episode": 1, "total": 7, "seed": 1601, "seat": 0, "opponent": "livestock_fertilizer", "arm": "coordinated", "coins_delta": 0.0, "match_delta": 0.0}
{"utc": "2026-09-12T00:20:11.185828+00:00", "stage": "EPISODE_CHECKPOINT_SAVED", "episode": 2, "total": 7, "seed": 1601, "seat": 0, "opponent": "livestock_fertilizer", "arm": "sequential", "coins_delta": 0.0, "match_delta": 0.0}
{"utc": "2026-09-12T00:20:12.189960+00:00", "stage": "EPISODE_CHEC

## Actual endpoint measurements
A completed report can contain a negative or null result. Read the decision; do not infer success from completion.

In [7]:
report = json.loads((OUT/'report.json').read_text())
print('DECISION:', report['decision'])
print('New research transitions:', report['new_research_interpreter_calls'])
print('Official submission score:', report['official_submission_score'])
print('Primary groups:', report['primary_pairs'], '| Negative controls:', report['negative_controls'])
display(pd.read_csv(OUT/'final_comparisons.csv'))
new_figures = style(experiment_figures(OUT))

DECISION: PROMISING_DEVELOPMENT_ONLY_FRESH_VALIDATION_REQUIRED
New research transitions: 14
Official submission score: None
Primary groups: 3 | Negative controls: 4


,seed,seat,opponent,arm,role,prefix_action_parity_checks,source_final_reward_parity,final_market_changed,shared_final_state_sha256,coins_control,...,opponent_coins_delta,coin_margin_control,coin_margin_guarded,coin_margin_delta,local_match_score_control,local_match_score_guarded,local_match_score_delta,residual_product_units_control,residual_product_units_guarded,residual_product_units_delta
0,1601,0,livestock_fertilizer,coordinated,primary,22,True,False,9317f5360a2d58de460bb7603f2951cdb3d945864512d1...,44840.0,...,0.0,758.0,758.0,0.0,1.0,1.0,0.0,0,0,0
1,1601,0,livestock_fertilizer,sequential,negative_control,22,True,False,2ae7bd400a0e6204d938e482f87a9af806ec3ca5dc04ae...,44837.0,...,0.0,759.0,759.0,0.0,1.0,1.0,0.0,0,0,0
2,1601,1,livestock_fertilizer,coordinated,primary,22,True,False,8fb1ee965c60a961e1e2bfca0c8104bfbb3c947638b9e7...,44840.0,...,0.0,758.0,758.0,0.0,1.0,1.0,0.0,0,0,0
3,1601,1,livestock_fertilizer,sequential,negative_control,22,True,False,88deec03aa4a3e2083b31f45f1bc1ddc5f2e5cc5d3c090...,44837.0,...,0.0,759.0,759.0,0.0,1.0,1.0,0.0,0,0,0
4,1602,0,livestock_fertilizer,coordinated,primary,22,True,True,fc45b93be018e984e50d1e9eb1a8e44a7718ec59551a71...,51764.0,...,0.0,2115.0,2157.0,42.0,1.0,1.0,0.0,1,0,-1
5,1602,0,livestock_fertilizer,sequential,negative_control,22,True,False,d5a1c290494b9e082913546b350beaf09d263d43556893...,50872.0,...,0.0,1228.0,1228.0,0.0,1.0,1.0,0.0,14,14,0
6,1602,1,livestock_fertilizer,sequential,negative_control,22,True,False,512aa7bfdfb343758d1f6c737557697b4e63300b92ea9e...,50102.0,...,0.0,-47.0,-47.0,0.0,0.0,0.0,0.0,17,17,0


In [8]:
new_figures[0].show()

In [9]:
new_figures[1].show()

## Conditional demand representation
Holding premiums assume current visible shops and no competing future trades. They are not measured future prices or a fitted ranking. Unavailable future-sale horizons are masked.

In [10]:
new_figures[2].show()

In [11]:
new_figures[3].show()

In [12]:
new_figures[4].show()
display(pd.read_csv(OUT/'feature_registry.csv').groupby('distinct_values').size().rename('candidate_count').reset_index())

,distinct_values,candidate_count
0,1,25
1,2,29
2,3,16
3,4,15
4,5,10
5,6,6
6,7,2
7,8,1
8,9,5
9,10,3


## Checkpoint and decision

Keep a negative/null result; do not continue merely because a run is cheap. A positive endpoint comparison remains selected development evidence from very few groups and one related opponent. Fresh groups, different opponents and hosted submission acceptance remain separate gates.

Press Ctrl+S before creating the return bundle. In a terminal use the commands below; after downloading the results, stop the JupyterLab application without deleting the space.

In [13]:
save_dashboard(figures + new_figures, OUT/'sale_timing_dashboard.html')
print('NOTEBOOK10_COMPLETE:', report['decision'])
print('Save this notebook with Ctrl+S, then run:')
print(f'cd "{BASE}"')
print(f'"{sys.executable}" run_timing.py bundle')
print('Return kaggriculture_sale_timing_results.zip, not the input ZIP.')

NOTEBOOK10_COMPLETE: PROMISING_DEVELOPMENT_ONLY_FRESH_VALIDATION_REQUIRED
Save this notebook with Ctrl+S, then run:
cd "/home/sagemaker-user/kaggriculture_sale_timing"
"/home/sagemaker-user/projects/kaggriculture/.venv/bin/python" run_timing.py bundle
Return kaggriculture_sale_timing_results.zip, not the input ZIP.
